# 1. Data Description and Pre-Processing
This section involves cleaning the dataset, handling missing values using imputation, encoding categorical features, scaling numerical features, and splitting the data into training, validation, and test sets.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

%config InlineBackend.figure_format = "retina"


In [ ]:
auto = pd.read_csv('adverts.csv')


In [ ]:
auto.head(50)

In [ ]:
auto.info()

In [ ]:
auto.describe()

In [ ]:
# 1. Visualize & quantify missingness
plt.figure(figsize=(10, 6))
sns.heatmap(auto.isnull(), cbar=False, cmap='viridis')
plt.title("Missing Value Heatmap")
plt.show()

missing_pct = auto.isnull().mean() * 100
print("Missing % by column:\n", missing_pct[missing_pct > 0].sort_values(ascending=False))

# 2. Create missingness indicator flags
for col in auto.columns:
    if auto[col].isnull().any():
        auto[f"{col}_missing"] = auto[col].isnull().astype(int)

# 3. Drop columns >50% missing (if still any)
threshold = len(auto) * 0.5
auto = auto.dropna(thresh=threshold, axis=1)

# 4. Numeric imputation: KNN
from sklearn.impute import KNNImputer
num_cols = auto.select_dtypes(include='number').drop(columns=['price'], errors='ignore').columns
knn = KNNImputer(n_neighbors=5)
auto[num_cols] = knn.fit_transform(auto[num_cols])

# 5. Categorical imputation: group-wise mode then global fallback
cat_cols = auto.select_dtypes(include='object').columns
for col in cat_cols:
    if 'model' in auto.columns:
        auto[col] = auto.groupby('model')[col] \
                       .transform(lambda x: x.fillna(x.mode()[0]) if not x.mode().empty else x.fillna("Unknown"))
    else:
        auto[col].fillna(auto[col].mode()[0], inplace=True)


In [ ]:
# Using IQR to detect and cap outliers
numeric_cols = auto.select_dtypes(include='number').columns

for col in numeric_cols:
    Q1 = auto[col].quantile(0.25)
    Q3 = auto[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    auto[col] = auto[col].clip(lower, upper)


In [ ]:
# Reduce cardinality: keep top N categories, label rest as "Other"
def reduce_cardinality(col, top_n=10):
    top = col.value_counts().index[:top_n]
    return col.where(col.isin(top), other='Other')

cat_cols = auto.select_dtypes(include='object').columns

for col in cat_cols:
    auto[col] = reduce_cardinality(auto[col])

# One-hot encoding
df_encoded = pd.get_dummies(auto, drop_first=True)
df_encoded.drop(columns=['public_reference'], inplace=True, errors='ignore')


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_encoded)

df_scaled = pd.DataFrame(scaled_data, columns=df_encoded.columns)


In [ ]:
from sklearn.model_selection import train_test_split

# Assume 'price' is the target column
X = df_scaled.drop(columns=['price'])
y = df_scaled['price']

# Split into train, validation, and test sets (60/20/20)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train shape: {X_train.shape}, Validation shape: {X_val.shape}, Test shape: {X_test.shape}")


# 2. Automated Feature Selection
Recursive Feature Elimination (RFE) was applied using a Random Forest estimator to select the most predictive 15 features, balancing performance with model simplicity.

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestRegressor

# Use the training data only for feature selection
estimator = RandomForestRegressor(n_estimators=100, random_state=42)

# Recursive Feature Elimination to select top N features
selector = RFE(estimator, n_features_to_select=15, step=1)  # adjust number as needed
selector = selector.fit(X_train, y_train)

# Create a dataframe showing selected features
selected_features = X_train.columns[selector.support_]
ranking = pd.Series(selector.ranking_, index=X_train.columns)

print("Selected features:\n", selected_features)
print("\nFeature Ranking:\n", ranking.sort_values())

# Reduce X to selected features only
X_train_selected = selector.transform(X_train)
X_val_selected = selector.transform(X_val) 
X_test_selected = selector.transform(X_test)


# 3. Tree Ensemble Models
Random Forest and Gradient Boosting Regressors were trained on the selected features to model non-linear patterns and assess their performance using R² and MSE.

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Train Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train_selected, y_train)

# Train Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
gb.fit(X_train_selected, y_train)

# Evaluate on Validation Set
models = {'Random Forest': rf, 'Gradient Boosting': gb}

for name, model in models.items():
    y_pred = model.predict(X_val_selected)
    mse = mean_squared_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    print(f"{name} - MSE: {mse:.2f}, R²: {r2:.4f}")


In [ ]:
import matplotlib.pyplot as plt

# Plot for one of the models (e.g., Gradient Boosting)
plt.figure(figsize=(6, 6))
plt.scatter(y_val, gb.predict(X_val_selected), alpha=0.3)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--')
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Gradient Boosting - Predicted vs Actual")
plt.grid(True)
plt.tight_layout()
plt.show()


# 4. Ensemble of Tree Ensembles
The predictions from Random Forest and Gradient Boosting were combined using simple averaging and stacked generalisation, with the stacked model yielding the best overall R².

In [ ]:
import numpy as np

# Average predictions from RF and GB
rf_preds = rf.predict(X_val_selected)
gb_preds = gb.predict(X_val_selected)

avg_preds = (rf_preds + gb_preds) / 2

# Evaluate Averaged Ensemble
from sklearn.metrics import mean_squared_error, r2_score

mse_avg = mean_squared_error(y_val, avg_preds)
r2_avg = r2_score(y_val, avg_preds)

print(f"Averaging Ensemble - MSE: {mse_avg:.2f}, R²: {r2_avg:.4f}")


In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

# Define base learners and meta learner
base_learners = [
    ('rf', RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)),
    ('gb', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42))
]
meta_learner = Ridge(alpha=1.0)

stacked_model = StackingRegressor(estimators=base_learners, final_estimator=meta_learner, cv=5)
stacked_model.fit(X_train_selected, y_train)

# Evaluate on validation set
stacked_preds = stacked_model.predict(X_val_selected)
mse_stack = mean_squared_error(y_val, stacked_preds)
r2_stack = r2_score(y_val, stacked_preds)

print(f"Stacking Ensemble - MSE: {mse_stack:.2f}, R²: {r2_stack:.4f}")


# 5. Feature Importance
Permutation importance was used to identify which features had the most impact on the Gradient Boosting model’s performance. Mileage, year of registration, and body type were most influential.

In [ ]:
from sklearn.inspection import permutation_importance


perm_result = permutation_importance(gb, X_val_selected, y_val, n_repeats=10, random_state=42)

# Organize results
perm_importances = pd.Series(perm_result.importances_mean, index=X_val.columns[selector.support_])
perm_importances.sort_values(ascending=False).plot(kind='barh', figsize=(8, 6))
plt.title("Permutation Feature Importance")
plt.xlabel("Mean Importance Score")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


# 6. SHAP and PDP Model Explanations
SHAP values and Partial Dependence Plots were used to visualise and explain how key features influenced the model’s predictions at both global and local levels.

In [ ]:
import shap

# Create explainer for Gradient Boosting
explainer = shap.Explainer(gb.predict, X_train_selected)
shap_values = explainer(X_val_selected)

# SHAP Summary Plot
shap.summary_plot(shap_values, features=X_val_selected, feature_names=X_val.columns[selector.support_])


In [ ]:
X_val_df = pd.DataFrame(X_val_selected, columns=X_val.columns[selector.support_])

# Create the explainer and disable additivity check
explainer = shap.Explainer(gb.predict, X_train_selected)
shap_values = explainer(X_val_selected)

# Global summary plot
shap.summary_plot(shap_values, features=X_val_selected, feature_names=X_val.columns[selector.support_])


In [ ]:
selected_features = X_train.columns[selector.support_].tolist()
print(selected_features)


In [ ]:
from sklearn.inspection import PartialDependenceDisplay

# Choose top features from SHAP or permutation importance
top_features = selected_features[:3]

# PDP plots for Gradient Boosting
PartialDependenceDisplay.from_estimator(
    gb, 
    X_val_df, 
    features=top_features,
    feature_names=X_val_df.columns,
    kind='average', 
    n_jobs=-1,
    grid_resolution=50
)
plt.tight_layout()
plt.show()


# 7. Dimensionality Reduction (Linear – PCA)
Principal Component Analysis was used to compress the feature space while retaining 95% of variance. Although effective in compression, predictive performance declined slightly.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# Fit PCA (start with enough components to explain ~95% variance)
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_selected)
X_val_pca = pca.transform(X_val_selected)
X_test_pca = pca.transform(X_test_selected)

# Check explained variance
explained_var = np.cumsum(pca.explained_variance_ratio_)
plt.figure(figsize=(8,5))
plt.plot(explained_var, marker='o')
plt.xlabel('Number of PCA Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Explained Variance by PCA Components')
plt.grid(True)
plt.show()

print(f"Number of components to explain 95% variance: {pca.n_components_}")


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Train models on PCA features
rf_pca = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
gb_pca = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)

rf_pca.fit(X_train_pca, y_train)
gb_pca.fit(X_train_pca, y_train)

# Evaluate on Validation set (PCA features)
for name, model in {'Random Forest (PCA)': rf_pca, 'Gradient Boosting (PCA)': gb_pca}.items():
    preds = model.predict(X_val_pca)
    mse = mean_squared_error(y_val, preds)
    r2 = r2_score(y_val, preds)
    print(f"{name}: MSE = {mse:.2f}, R² = {r2:.4f}")


# 8. Dimensionality Reduction (Non-Linear – Isomap)
Isomap was used to visualise complex non-linear relationships. A Gradient Boosting model trained on the 2D Isomap-transformed features showed reduced predictive power (R² ≈ 0.30).

In [ ]:
from sklearn.manifold import Isomap

# Fit Isomap (2D for visualization)
isomap = Isomap(n_components=2)
X_train_iso = isomap.fit_transform(X_train_selected)

# Plot to visualize clusters/patterns
plt.figure(figsize=(8,6))
plt.scatter(X_train_iso[:,0], X_train_iso[:,1], c=y_train, cmap='viridis', s=15, alpha=0.7)
plt.colorbar(label='Price')
plt.title("Isomap (2D) embedding of Training Data")
plt.xlabel("Isomap Feature 1")
plt.ylabel("Isomap Feature 2")
plt.grid(True)
plt.show()




In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Fit model on Isomap-transformed training set
gb_iso = GradientBoostingRegressor(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
gb_iso.fit(X_train_iso, y_train)

# Predict and evaluate
# Fit Isomap on training and transform validation separately
X_val_iso = isomap.fit_transform(X_val_selected)
y_pred_iso = gb_iso.predict(X_val_iso)
mse_iso = mean_squared_error(y_val, y_pred_iso)
r2_iso = r2_score(y_val, y_pred_iso)
print(f"Gradient Boosting (Isomap): MSE = {mse_iso:.2f}, R² = {r2_iso:.4f}")


# 9. Polynomial Regression
Second-degree polynomial regression with Ridge regularisation captured some non-linear relationships and achieved a validation R² of 0.6905, lower than tree ensembles but useful as a linear benchmark.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

# Simple polynomial regression pipeline
poly_model = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0))
])

poly_model.fit(X_train_selected, y_train)

# Evaluation
poly_preds = poly_model.predict(X_val_selected)
print(f"Polynomial Regression: R² = {r2_score(y_val, poly_preds):.4f}")


# 10. Clustering for Feature Engineering
KMeans clustering was applied to generate a new categorical feature. The Gradient Boosting model trained with this new feature achieved an R² of 0.7068, showing mild improvement over other non-ensemble methods.

In [ ]:
from sklearn.cluster import KMeans

# Create clusters on training data
kmeans = KMeans(n_clusters=5, random_state=42)
cluster_labels = kmeans.fit_predict(X_train_selected)

# Add cluster labels as a new feature
X_train_clustered = np.column_stack((X_train_selected, cluster_labels))
X_val_cluster_labels = kmeans.predict(X_val_selected)
X_val_clustered = np.column_stack((X_val_selected, X_val_cluster_labels))

# Retrain one model (e.g., GB) to test improvement
gb_clustered = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_clustered.fit(X_train_clustered, y_train)

cluster_preds = gb_clustered.predict(X_val_clustered)
print(f"GB with clustering: R² = {r2_score(y_val, cluster_preds):.4f}")


# Model Performance Comparison (Validation R²)

This cell generates a horizontal bar chart comparing the R² scores of all trained models on the validation set. The scores are calculated dynamically using `r2_score` applied to each model’s predictions. This visual summary highlights the relative performance of:

- Tree-based models (Random Forest, Gradient Boosting)
- Ensemble techniques (Averaging, Stacking)
- Dimensionality reduction approaches (PCA, Isomap)
- Polynomial regression
- Clustering-based feature engineering

It provides a quick reference for evaluating how different modelling strategies impacted prediction accuracy.


In [ ]:
# Dynamically compute R² from predictions
r2_rf = r2_score(y_val, rf_preds)
r2_gb = r2_score(y_val, gb_preds)
r2_avg = r2_score(y_val, avg_preds)
r2_stack = r2_score(y_val, stacked_preds)
r2_pca_gb = r2_score(y_val, preds)
r2_iso_gb = r2_score(y_val, y_pred_iso)
r2_poly = r2_score(y_val, poly_preds)
r2_cluster = r2_score(y_val, cluster_preds)

# Define model names and corresponding scores
model_names = [
    "Random Forest", 
    "Gradient Boosting", 
    "Averaging Ensemble", 
    "Stacking Ensemble", 
    "PCA + GB", 
    "Isomap + GB", 
    "Polynomial Regression", 
    "GB + Clustering"
]

r2_scores = [
    r2_rf,
    r2_gb,
    r2_avg,
    r2_stack,
    r2_pca_gb,
    r2_iso_gb,
    r2_poly,
    r2_cluster
]

# Plotting
plt.figure(figsize=(12, 6))
bars = plt.barh(model_names, r2_scores, color='skyblue')
plt.xlabel("R² Score")
plt.title("Model Performance Comparison (Validation R²)")
plt.xlim(0, max(r2_scores) + 0.05)
plt.grid(axis="x", linestyle="--", alpha=0.7)

for bar, score in zip(bars, r2_scores):
    plt.text(score + 0.01, bar.get_y() + bar.get_height()/2, f"{score:.4f}", va='center')

plt.tight_layout()
plt.show()
